# 08 — Figuras de linguagem com GPT-5.6

Prepara codebook e piloto humano, compara modelos e gera produção estruturada por Batch API.

In [ ]:
from google.colab import drive

drive.mount("/content/drive")

In [ ]:
import os
import subprocess
import sys
from pathlib import Path

DATA_ROOT = Path("/content/drive/MyDrive/falando_nela/data")
REPO_DIR = Path("/content/falando_nela")
REPO_URL = "https://github.com/pedblan/falando_nela.git"
REPO_REF = ""  # Opcional: branch, tag ou commit; vazio acompanha o default remoto.

if not REPO_DIR.exists():
    subprocess.run(["git", "clone", REPO_URL, str(REPO_DIR)], check=True)
else:
    subprocess.run(["git", "-C", str(REPO_DIR), "fetch", "--all", "--tags", "--prune"], check=True)
    if not REPO_REF:
        subprocess.run(["git", "-C", str(REPO_DIR), "pull", "--ff-only"], check=True)
if REPO_REF:
    subprocess.run(["git", "-C", str(REPO_DIR), "checkout", REPO_REF], check=True)

os.chdir(REPO_DIR)
os.environ["FALANDO_NELA_DATA_ROOT"] = str(DATA_ROOT)
subprocess.run(
    [
        sys.executable,
        "-m",
        "pip",
        "install",
        "--upgrade",
        "--force-reinstall",
        "--no-cache-dir",
        "numpy==2.0.2",
        "pandas==2.2.3",
    ],
    check=True,
)
subprocess.run([sys.executable, "-m", "pip", "install", "-r", "requirements-analise.txt"], check=True)
ABI_CHECK = subprocess.run(
    [
        sys.executable,
        "-c",
        (
            "import numpy as np; import pandas as pd; "
            "assert np.__version__ == '2.0.2', np.__version__; "
            "assert pd.__version__ == '2.2.3', pd.__version__; "
            "print(f'NumPy {np.__version__}; pandas {pd.__version__}')"
        ),
    ],
    check=True,
    text=True,
    capture_output=True,
)
import numpy as np
import pandas as pd

assert np.__version__ == "2.0.2", f"Reinicie a sessao do Colab: NumPy carregado={np.__version__}"
assert pd.__version__ == "2.2.3", f"Reinicie a sessao do Colab: pandas carregado={pd.__version__}"
print("Data root:", DATA_ROOT)
print("Commit:", subprocess.run(["git", "rev-parse", "HEAD"], check=True, text=True, capture_output=True).stdout.strip())
print("ABI:", ABI_CHECK.stdout.strip())

## Configuração

Use o mesmo `RUN_ID` em toda a suíte. A configuração versionada é a fonte de verdade.

In [ ]:
from analise.discursos_plenario.config import load_config, resolve_input_paths, resolve_output_root

RUN_ID = "analise-plenario-20260717-v1"
CONFIG_PATH = REPO_DIR / "analise" / "discursos_plenario" / "config.v1.json"
ANALYSIS_CONFIG = load_config(CONFIG_PATH)
RUN_OUTPUT_ROOT = resolve_output_root(ANALYSIS_CONFIG, DATA_ROOT, RUN_ID)
INPUT_PATHS = resolve_input_paths(ANALYSIS_CONFIG, DATA_ROOT)
RODAR_ETAPA = False

assert ANALYSIS_CONFIG.date_start == "2010-02-02"
assert ANALYSIS_CONFIG.date_end == "2026-07-13"
assert ANALYSIS_CONFIG.raw["complete_year_end"] == 2025
assert ANALYSIS_CONFIG.raw["ytd_year"] == 2026
print("Run:", RUN_ID)
print("Saida:", RUN_OUTPUT_ROOT)

## Decisão metodológica

GPT-5.6 Sol é o padrão. Luna ou Terra só podem substituí-lo após não inferioridade pareada contra o mesmo piloto humano adjudicado.

In [ ]:
FIGURAS_SNAPSHOT_PATH = RUN_OUTPUT_ROOT / "00_snapshot" / "discursos_plenario_snapshot.parquet"
assert FIGURAS_SNAPSHOT_PATH.exists(), "Execute o caderno 00."
FIGURAS_MODELS = ANALYSIS_CONFIG.raw["openai"]["figures_candidate_models"]
FIGURAS_DEFAULT_MODEL = ANALYSIS_CONFIG.raw["openai"]["figures_default_model"]
FIGURAS_SAMPLE_LIMIT = None
GERAR_JSONL = False
ENVIAR_BATCH = False
BAIXAR_BATCH_FIGURAS = False
PROCESSAR_BATCH_FIGURAS = False
FIGURAS_MODEL_FOR_BATCH = FIGURAS_DEFAULT_MODEL
FIGURAS_BATCH_SCOPE = "piloto"  # piloto | producao
print("Modelos do piloto:", FIGURAS_MODELS)

## Execução

A etapa cara permanece desativada até a inspeção das entradas e dos parâmetros acima.

In [ ]:
from analise.discursos_plenario.figuras import prepare_figures_stage

FIGURAS_SETUP_RESULT = None
if RODAR_ETAPA:
    FIGURAS_SETUP_RESULT = prepare_figures_stage(
        data_root=DATA_ROOT,
        run_id=RUN_ID,
        config_path=CONFIG_PATH,
        sample_limit=FIGURAS_SAMPLE_LIMIT,
    )
    print(FIGURAS_SETUP_RESULT["manifest_path"])
else:
    print("Setup não executado. Complete codebook e piloto antes do Batch.")

## Validação imediata

Esta checagem não substitui os testes sintéticos nem a revisão dos manifests.

In [ ]:
import pandas as pd

FIGURAS_CODEBOOK_PATH = RUN_OUTPUT_ROOT / "08_figuras" / "codebook.csv"
FIGURAS_PILOT_PATH = RUN_OUTPUT_ROOT / "08_figuras" / "piloto_humano.csv"
if FIGURAS_CODEBOOK_PATH.exists():
    FIGURAS_CODEBOOK = pd.read_csv(FIGURAS_CODEBOOK_PATH)
    assert set(FIGURAS_CODEBOOK["categoria"]) == set(ANALYSIS_CONFIG.raw["rhetorical_figures"])
    display(FIGURAS_CODEBOOK)

## Preparar o JSONL do Batch

A geração exige codebook preenchido. Crie arquivos separados para cada modelo do piloto e para a produção escolhida.

In [ ]:
import pandas as pd
from analise.discursos_plenario.figuras import write_batch_jsonl

FIGURAS_BATCH_REQUEST_PATH = RUN_OUTPUT_ROOT / "08_figuras" / f"batch_{FIGURAS_MODEL_FOR_BATCH}.jsonl"
if GERAR_JSONL:
    FIGURAS_CODEBOOK_READY = pd.read_csv(FIGURAS_CODEBOOK_PATH).fillna("")
    FIGURAS_CODEBOOK_FIELDS = ["definicao_operacional", "criterio_positivo", "criterio_negativo", "caso_limitrofe"]
    assert FIGURAS_CODEBOOK_READY[FIGURAS_CODEBOOK_FIELDS].apply(lambda column: column.str.strip().ne("").all()).all(), "Complete o codebook."
    assert FIGURAS_BATCH_SCOPE in {"piloto", "producao"}
    FIGURAS_SAMPLE_FILENAME = "amostra_piloto.parquet" if FIGURAS_BATCH_SCOPE == "piloto" else "amostra_elegivel.parquet"
    FIGURAS_SAMPLE = pd.read_parquet(RUN_OUTPUT_ROOT / "08_figuras" / FIGURAS_SAMPLE_FILENAME)
    FIGURAS_CODEBOOK_TEXT = FIGURAS_CODEBOOK_READY.to_csv(index=False)
    write_batch_jsonl(
        FIGURAS_SAMPLE,
        FIGURAS_BATCH_REQUEST_PATH,
        codebook=FIGURAS_CODEBOOK_TEXT,
        config=ANALYSIS_CONFIG,
        model=FIGURAS_MODEL_FOR_BATCH,
    )
    print(FIGURAS_BATCH_SCOPE, FIGURAS_MODEL_FOR_BATCH, len(FIGURAS_SAMPLE), FIGURAS_BATCH_REQUEST_PATH)
else:
    print("JSONL não gerado.")

## Enviar o Batch explicitamente

A chave é lida do ambiente ou dos Secrets do Colab e não é persistida. Guarde o Batch ID no arquivo de controle.

In [ ]:
import os
from openai import OpenAI
from analise.discursos_plenario.figuras import submit_responses_batch
from analise.discursos_plenario.io import write_json_atomic

FIGURAS_BATCH_SUBMISSION = None
if ENVIAR_BATCH:
    assert FIGURAS_BATCH_REQUEST_PATH.exists(), "Gere e inspecione o JSONL primeiro."
    if not os.environ.get("OPENAI_API_KEY"):
        try:
            from google.colab import userdata
            FIGURAS_SECRET = userdata.get("OPENAI_API_KEY")
        except Exception:
            FIGURAS_SECRET = None
        if FIGURAS_SECRET:
            os.environ["OPENAI_API_KEY"] = FIGURAS_SECRET
    assert os.environ.get("OPENAI_API_KEY"), "Configure OPENAI_API_KEY no ambiente ou nos Secrets do Colab."
    FIGURAS_CLIENT = OpenAI()
    FIGURAS_BATCH_SUBMISSION = submit_responses_batch(
        FIGURAS_CLIENT,
        FIGURAS_BATCH_REQUEST_PATH,
        description=f"{RUN_ID}:{FIGURAS_MODEL_FOR_BATCH}",
    )
    FIGURAS_BATCH_CONTROL = {
        "batch_id": FIGURAS_BATCH_SUBMISSION.id,
        "model": FIGURAS_MODEL_FOR_BATCH,
        "request_path": str(FIGURAS_BATCH_REQUEST_PATH),
    }
    write_json_atomic(RUN_OUTPUT_ROOT / "08_figuras" / f"batch_{FIGURAS_MODEL_FOR_BATCH}.json", FIGURAS_BATCH_CONTROL)
    print("Batch criado:", FIGURAS_BATCH_SUBMISSION.id)
else:
    print("Envio desativado.")

## Avaliar concordância e não inferioridade

Depois de reconciliar as respostas, compare todos os modelos nos mesmos discursos e oradores. Defina a margem antes de olhar o resultado.

In [ ]:
from analise.discursos_plenario.figuras import compare_models_against_human

FIGURAS_NONINFERIORITY_MARGIN = 0.03
FIGURAS_EVALUATION_READY = False
FIGURAS_MODEL_SUMMARY = None
FIGURAS_MODEL_COMPARISONS = None
if FIGURAS_EVALUATION_READY:
    FIGURAS_MODEL_SUMMARY, FIGURAS_MODEL_COMPARISONS = compare_models_against_human(
        FIGURAS_HUMAN_LONG,
        FIGURAS_RESULTS_LONG,
        FIGURAS_METADATA,
        ANALYSIS_CONFIG.raw["rhetorical_figures"],
        reference_model=FIGURAS_DEFAULT_MODEL,
        noninferiority_margin=FIGURAS_NONINFERIORITY_MARGIN,
        repetitions=ANALYSIS_CONFIG.raw["bootstrap_repetitions"],
        seed=ANALYSIS_CONFIG.seed,
    )
    display(FIGURAS_MODEL_SUMMARY)
    display(FIGURAS_MODEL_COMPARISONS)
else:
    print("Carregue piloto humano e resultados reconciliados antes de habilitar a avaliação.")

## Baixar e consolidar o Batch de figuras

A consolidação reconcilia `custom_id`, calcula prevalência por mil palavras, avalia o piloto adjudicado e estima custo somente se a tabela oficial de preços estiver preenchida.

In [ ]:
import json
import os
from openai import OpenAI
from analise.discursos_plenario.figuras import download_completed_batch, run_figures_results

FIGURAS_BATCH_CONTROL_PATH = RUN_OUTPUT_ROOT / "08_figuras" / f"batch_{FIGURAS_MODEL_FOR_BATCH}.json"
FIGURAS_BATCH_OUTPUT_PATH = RUN_OUTPUT_ROOT / "08_figuras" / f"batch_{FIGURAS_MODEL_FOR_BATCH}_output.jsonl"
if BAIXAR_BATCH_FIGURAS:
    assert FIGURAS_BATCH_CONTROL_PATH.exists(), FIGURAS_BATCH_CONTROL_PATH
    if not os.environ.get("OPENAI_API_KEY"):
        try:
            from google.colab import userdata
            FIGURAS_DOWNLOAD_SECRET = userdata.get("OPENAI_API_KEY")
        except Exception:
            FIGURAS_DOWNLOAD_SECRET = None
        if FIGURAS_DOWNLOAD_SECRET:
            os.environ["OPENAI_API_KEY"] = FIGURAS_DOWNLOAD_SECRET
    assert os.environ.get("OPENAI_API_KEY"), "Configure OPENAI_API_KEY no ambiente ou nos Secrets do Colab."
    FIGURAS_DOWNLOAD_CLIENT = OpenAI()
    FIGURAS_BATCH_CONTROL_LOADED = json.loads(FIGURAS_BATCH_CONTROL_PATH.read_text(encoding="utf-8"))
    download_completed_batch(
        FIGURAS_DOWNLOAD_CLIENT,
        FIGURAS_BATCH_CONTROL_LOADED["batch_id"],
        FIGURAS_BATCH_OUTPUT_PATH,
    )
    print(FIGURAS_BATCH_OUTPUT_PATH)
FIGURAS_RESULTS_MANIFEST = None
if PROCESSAR_BATCH_FIGURAS:
    assert FIGURAS_BATCH_OUTPUT_PATH.exists(), FIGURAS_BATCH_OUTPUT_PATH
    FIGURAS_RESULTS_MANIFEST = run_figures_results(
        data_root=DATA_ROOT,
        run_id=RUN_ID,
        batch_output_path=FIGURAS_BATCH_OUTPUT_PATH,
        request_path=FIGURAS_BATCH_REQUEST_PATH,
        model=FIGURAS_MODEL_FOR_BATCH,
        config_path=CONFIG_PATH,
    )
    print(FIGURAS_RESULTS_MANIFEST["manifest_path"])
else:
    print("Consolidação da saída desativada.")